# ResNet50 + Optical Flow (CASME II)

Notebook para entrenamiento de microexpresiones usando secuencias `.npy` (optical flow) y ResNet50.

Ajusta rutas antes de ejecutar.

In [1]:
'''import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import csv
from pathlib import Path'''
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import csv
from pathlib import Path

In [ ]:
from pathlib import Path
import torch

# CONFIG
BASE_DIR = Path("Microexpresiones")

# ── Fuentes PRIMARIAS: se usan para train Y val ──────────────────────────
PRIMARY_DATA_DIRS = [
    "../outputs_dataset_index",
    "../output_extraction_smic"
]
PRIMARY_CSV_INDEXES = [
    "../outputs_dataset_index/extraction_index.csv",
    "../output_extraction_smic/extraction_index.csv"
]
PRIMARY_SOURCE_FILTERS = [
    None,                           # CASME II: todas las emociones
    {"sorpresa", "surprise"},       # SMIC: acepta tanto español como inglés
]

# ── Fuente MEME: solo para balancear clases con pocas muestras en train ──
MEME_DATA_DIR  = "../output_extraction_meme_v2"
MEME_CSV_INDEX = "../output_extraction_meme_v2/extraction_index.csv"

# MIN_TRAIN_SAMPLES=30: con 50 se añaden 23 MEME para felicidad (25prim → 48% MEME)
# → domain shift grave: el modelo aprende MEME-felicidad, val es CASME-felicidad
# Con 30: solo +5 MEME para felicidad, cero para sorpresa (35 > 30 ya) → sin shift
MIN_TRAIN_SAMPLES = 30
MAX_MEME_RATIO    = 5.0

# ── Cap de peso de clase: evita que clases raras dominen el gradiente ──
MAX_CLASS_WEIGHT = 8.0

NUM_FRAMES = 16
BATCH_SIZE = 8
EPOCHS     = 100
LR         = 3e-4
LR_LAYER4  = 1e-5     # layer4 descongelada todo el entrenamiento

N_FOLDS = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print(f"\nFuentes PRIMARIAS ({len(PRIMARY_DATA_DIRS)}):")
for i, d in enumerate(PRIMARY_DATA_DIRS):
    f = PRIMARY_SOURCE_FILTERS[i]
    print(f"  [{i}] {d}  →  filtro: {f if f else 'todas las emociones'}")
print(f"\nFuente MEME (solo balanceo de train): {MEME_DATA_DIR}")
print(f"Umbral mínimo por fold de train  : MIN_TRAIN_SAMPLES = {MIN_TRAIN_SAMPLES}")
print(f"Ratio máximo MEME/primario       : MAX_MEME_RATIO = {MAX_MEME_RATIO}x")
print(f"Cap máximo peso de clase         : MAX_CLASS_WEIGHT = {MAX_CLASS_WEIGHT}x")
print(f"Cross-validation                 : {N_FOLDS}-Fold estratificado")
print(f"EPOCHS={EPOCHS}  LR={LR}  LR_LAYER4={LR_LAYER4}  (layer4 descongelada todo el entrenamiento)")


Using device: cuda

Fuentes PRIMARIAS (2):
  [0] ../outputs_dataset_index  →  filtro: todas las emociones
  [1] ../output_extraction_smic  →  filtro: {'sorpresa'}

Fuente MEME (solo balanceo de train): ../output_extraction_meme_correction
Umbral mínimo por fold de train  : MIN_TRAIN_SAMPLES = 30
Ratio máximo MEME/primario       : MAX_MEME_RATIO = 3.0x
  → Clases con pocos primarios no quedarán dominadas por ruido MEME
Cross-validation                 : 5-Fold estratificado


In [3]:
from PIL import Image

# ──────────────────────────────────────────────
# VAL / TEST: solo normalización, sin aleatoriedad
# ──────────────────────────────────────────────
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ──────────────────────────────────────────────
# TRAIN: augmentación segura para optical flow
#
#  ✅ RandomResizedCrop — robustez espacial leve
#  ✅ RandomErasing     — simula oclusiones parciales de la cara
#  ❌ NO HorizontalFlip  — invertiría dx sin corregir el signo del flujo
#  ❌ NO ColorJitter(saturation/hue) — cambiaría la dirección del flujo
# ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.08), value=0),
])

# 'transform' sigue usándose en FlowDataset como val (base)
transform = val_transform

print("✅ val_transform  : Resize(224) + Normalize")
print("✅ train_transform: RandomResizedCrop + RandomErasing")
print("   ⚠️  Flip y ColorJitter omitidos (corrompen dirección del flujo)")

✅ val_transform  : Resize(224) + Normalize
✅ train_transform: RandomResizedCrop + RandomErasing
   ⚠️  Flip y ColorJitter omitidos (corrompen dirección del flujo)


In [ ]:
class FlowDataset(Dataset):
    def __init__(self, data_dirs, csv_files, transform, num_frames=16, source_filters=None):
        self.transform = transform
        self.num_frames = num_frames
        self.samples = []  # (path, label, subject)

        if source_filters is None:
            source_filters = [None] * len(data_dirs)

        label_alias = {
            "felicidad": "felicidad", "feliz":   "felicidad",
            "enojo":     "enojo",     "ira":      "enojo",
            "miedo":     "miedo",
            "tristeza":  "tristeza",
            "sorpresa":  "sorpresa",
            "asco":      "asco",
            "represion": "otros",     "otros":    "otros",
            "happiness": "felicidad", "happy":    "felicidad",
            "anger":     "enojo",
            "fear":      "miedo",
            "sadness":   "tristeza",
            "surprise":  "sorpresa",
            "disgust":   "asco",
            "contempt":  "otros",     "neutral":  "neutral",
            "repression":"otros",
        }

        for data_dir, csv_index, allowed_raw in zip(data_dirs, csv_files, source_filters):
            data_dir = Path(data_dir)
            loaded = skipped = 0
            with open(csv_index, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    raw_label = row['emocion'].strip().lower()
                    if allowed_raw is not None and raw_label not in allowed_raw:
                        skipped += 1; continue
                    label = label_alias.get(raw_label)
                    if label is None:
                        skipped += 1; continue
                    rel     = row['archivo'].replace("\\", "/")
                    path    = data_dir / rel
                    subject = row.get('sujeto', row.get('persona', 'unknown'))
                    if path.exists():
                        self.samples.append((path, label, subject))
                        loaded += 1
            print(f"  {data_dir.name}: {loaded} cargadas"
                  + (f" ({skipped} omitidas)" if skipped > 0 else ""))

        self.labels    = sorted(set(s[1] for s in self.samples))
        self.label_map = {l: i for i, l in enumerate(self.labels)}
        print(f"\nEtiquetas finales: {self.label_map}")

    def __len__(self):
        return len(self.samples)

    def sample_frames(self, seq, jitter=False):
        """Muestrea num_frames de la secuencia con jitter temporal opcional."""
        n = seq.shape[0]
        if n == 0:
            raise ValueError("Secuencia vacía")
        idx = np.linspace(0, n - 1, self.num_frames, dtype=float)
        if jitter and n > 2:
            # Desplazamiento aleatorio de hasta ±15% del intervalo entre frames
            step = (n - 1) / max(self.num_frames - 1, 1)
            noise = np.random.uniform(-0.15 * step, 0.15 * step, size=self.num_frames)
            idx = np.clip(idx + noise, 0, n - 1)
        return seq[idx.astype(int)]

    def _get_raw_frames(self, idx, jitter=False):
        """Devuelve frames numpy sin convertir a tensor (para augmentación externa)."""
        path, label, _ = self.samples[idx]
        seq    = np.load(path)
        frames = self.sample_frames(seq, jitter=jitter)
        return frames, label

    def __getitem__(self, idx):
        frames, label = self._get_raw_frames(idx, jitter=False)
        return self._frames_to_tensor(frames), self.label_map[label]

    def _frames_to_tensor(self, frames, transform=None):
        tr = transform if transform is not None else self.transform
        imgs = []
        for f in frames:
            f0  = np.clip((f[..., 0] + 1) / 2, 0, 1)
            f1  = np.clip((f[..., 1] + 1) / 2, 0, 1)
            f2  = np.clip(f[..., 2], 0, 1)
            rgb = (np.stack([f0, f1, f2], axis=-1) * 255).astype(np.uint8)
            imgs.append(tr(Image.fromarray(rgb)))
        return torch.stack(imgs)


class AugmentedSubset(Dataset):
    """Subset que aplica train_transform + jitter temporal."""
    def __init__(self, base_dataset, indices, transform):
        self.base      = base_dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        frames, label = self.base._get_raw_frames(self.indices[idx], jitter=True)
        x = self.base._frames_to_tensor(frames, transform=self.transform)
        return x, self.base.label_map[label]


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class TemporalAttention(nn.Module):
    """Attention sobre la dimensión temporal: aprende qué frames son más discriminativos."""
    def __init__(self, feat_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, feat_seq):
        # feat_seq: (B, T, D)
        scores = self.attn(feat_seq)           # (B, T, 1)
        weights = torch.softmax(scores, dim=1) # (B, T, 1)
        return (feat_seq * weights).sum(dim=1) # (B, D)


class ResNet50FlowMLP(nn.Module):
    """
    ResNet50: conv1..layer3 congelados, layer4 descongelada con LR_LAYER4 muy bajo.
    Dropout 0.55/0.45 — experimentalmente mejor que 0.65/0.55 con este dataset.
    Con 0.65: TrAcc se estanca en 0.70, el modelo subajusta y F1 val empeora.
    """
    def __init__(self, num_classes, proj_dim=256):
        super().__init__()

        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # Backbone CONGELADO: conv1..layer3
        self.backbone_frozen = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3,
        )
        for param in self.backbone_frozen.parameters():
            param.requires_grad = False

        # layer4 DESCONGELADA — fine-tuning controlado con LR_LAYER4 = 1e-5
        self.layer4 = resnet.layer4
        for param in self.layer4.parameters():
            param.requires_grad = True

        # Pool espacial → (B*T, 2048)
        self.pool = nn.AdaptiveAvgPool2d(1)

        # Atención temporal (aprende qué frames son más informativos)
        self.temporal_attn = TemporalAttention(2048)

        # Clasificador MLP — dropout 0.55/0.45 (configuración que dio F1=0.55)
        self.classifier = nn.Sequential(
            nn.Linear(2048, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Dropout(0.55),
            nn.Linear(proj_dim, proj_dim // 2),
            nn.BatchNorm1d(proj_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.45),
            nn.Linear(proj_dim // 2, num_classes),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)

        with torch.no_grad():
            feat = self.backbone_frozen(x)

        feat = self.layer4(feat)

        feat = self.pool(feat).flatten(1)
        feat = feat.view(B, T, -1)

        feat = self.temporal_attn(feat)

        return self.classifier(feat)


In [ ]:
import csv
from pathlib import Path

DATA_DIRS = [
    Path("../outputs_dataset_index"),
    Path("../output_extraction_meme_v2")
]

CSV_INDEXES = [
    "../outputs_dataset_index/extraction_index.csv",
    "../output_extraction_meme_v2/extraction_index.csv"
]

for data_dir, csv_file in zip(DATA_DIRS, CSV_INDEXES):
    print(f"\n=== Revisando dataset: {data_dir} ===\n")

    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            rel = row['archivo'].replace("\\", "/")
            full = data_dir / rel

            print("CSV:", rel)
            print("FULL:", full)
            print("EXISTS:", full.exists())
            print("-"*50)

            if i == 5:
                break



=== Revisando dataset: ../outputs_dataset_index ===

CSV: Usuario_01/asco/EP19_05f.npy
FULL: ../outputs_dataset_index/Usuario_01/asco/EP19_05f.npy
EXISTS: True
--------------------------------------------------
CSV: Usuario_01/asco/EP19_06f.npy
FULL: ../outputs_dataset_index/Usuario_01/asco/EP19_06f.npy
EXISTS: True
--------------------------------------------------
CSV: Usuario_01/felicidad/EP02_01f.npy
FULL: ../outputs_dataset_index/Usuario_01/felicidad/EP02_01f.npy
EXISTS: True
--------------------------------------------------
CSV: Usuario_01/otros/EP03_02.npy
FULL: ../outputs_dataset_index/Usuario_01/otros/EP03_02.npy
EXISTS: True
--------------------------------------------------
CSV: Usuario_01/otros/EP04_02.npy
FULL: ../outputs_dataset_index/Usuario_01/otros/EP04_02.npy
EXISTS: True
--------------------------------------------------
CSV: Usuario_01/otros/EP04_03.npy
FULL: ../outputs_dataset_index/Usuario_01/otros/EP04_03.npy
EXISTS: True
---------------------------------------

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
from collections import Counter

# =========================
# LOAD DATA PRIMARIO (CASME II + SMIC)
# =========================
print("Cargando dataset PRIMARIO (CASME II + SMIC)...")
dataset = FlowDataset(
    data_dirs=PRIMARY_DATA_DIRS,
    csv_files=PRIMARY_CSV_INDEXES,
    transform=transform,
    num_frames=NUM_FRAMES,
    source_filters=PRIMARY_SOURCE_FILTERS
)

# Quitar "otros" — conservar todas las demás clases (incluyendo tristeza para el análisis)
dataset.samples = [s for s in dataset.samples if s[1] != "otros"]

# =========================
# PROMOVER CLASES ESCASAS de MEME_v2 → pool PRIMARIO
# =========================
_PROMOTE_FROM_MEME = {
    'neutral': 'neutral',
    'miedo':   'miedo',
    'fear':    'miedo',
}

print(f"\nCargando clases escasas desde MEME_v2 → pool primario...")
print(f"  Clases a promover: {sorted(set(_PROMOTE_FROM_MEME.values()))}")

_added_counts = {}
_meme_dir_p = Path(MEME_DATA_DIR)
with open(MEME_CSV_INDEX, 'r', encoding='utf-8') as _f:
    for _row in csv.DictReader(_f):
        _raw   = _row['emocion'].strip().lower()
        _label = _PROMOTE_FROM_MEME.get(_raw)
        if _label is None:
            continue
        _rel  = _row['archivo'].replace("\\", "/")
        _path = _meme_dir_p / _rel
        _subj = _row.get('sujeto', _row.get('persona', 'unknown'))
        if _path.exists():
            dataset.samples.append((_path, _label, _subj))
            _added_counts[_label] = _added_counts.get(_label, 0) + 1

for _lbl, _n in sorted(_added_counts.items()):
    print(f"  {_lbl}: {_n} muestras añadidas al primario")

_promoted_labels = set(_PROMOTE_FROM_MEME.values())

# Reconstruir label_map con TODAS las clases (incluyendo tristeza para el análisis)
labels_unique_full = sorted(set(s[1] for s in dataset.samples))
dataset.label_map  = {l: i for i, l in enumerate(labels_unique_full)}
_nc_full           = len(dataset.label_map)
_counts_full       = np.bincount(np.array([dataset.label_map[s[1]] for s in dataset.samples]))

print(f"\nClases detectadas ({_nc_full}): {dataset.label_map}")
for _lbl, _idx in dataset.label_map.items():
    print(f"  {_lbl}: {_counts_full[_idx]} muestras")

# =========================
# ANÁLISIS DE SEPARABILIDAD DE CLASES  (justificación académica)
#
# Flag para saltarlo en reruns si ya tienes los resultados.
# Ponlo en False para ir directo al entrenamiento.
# =========================
RUN_SEPARABILITY_ANALYSIS = True

if RUN_SEPARABILITY_ANALYSIS:
    print(f"\n{'='*60}")
    print(f"📊 ANÁLISIS DE SEPARABILIDAD DE CLASES")
    print(f"   Espacio: ResNet50 features (2048-dim, backbone+layer4 congelados)")
    print(f"   Métrica: Fisher Discriminant Ratio normalizado por dimensión")
    print(f"   Nota: usa 1 frame central por muestra — liviano en memoria")
    print(f"{'='*60}")

    from torchvision import models as _tv_models
    from PIL import Image as _PIL_Image

    # Extractor liviano: 1 frame por muestra → sin DataLoader ni numpy gigante en RAM
    class _FeatExtractor(nn.Module):
        def __init__(self):
            super().__init__()
            _r = _tv_models.resnet50(weights=_tv_models.ResNet50_Weights.DEFAULT)
            self.net  = nn.Sequential(
                _r.conv1, _r.bn1, _r.relu, _r.maxpool,
                _r.layer1, _r.layer2, _r.layer3, _r.layer4,
            )
            self.pool = nn.AdaptiveAvgPool2d(1)
            for p in self.parameters():
                p.requires_grad = False

        def forward(self, x):
            return self.pool(self.net(x)).flatten(1)  # (B, 2048)

    _ext = _FeatExtractor().to(DEVICE).eval()

    # ── Extrae 1 frame central por muestra (reduce carga 16× vs todos los frames) ──
    print(f"  ⏳ Extrayendo features ({len(dataset.samples)} muestras × 1 frame)...")

    _SEP_BATCH = 16   # imágenes a la vez — mucho menos que los 128 anteriores
    _feat_buf, _lab_buf = [], []
    _img_buf  = []

    def _frame_to_tensor_single(frame_np, tr=val_transform):
        """Convierte un frame numpy (H,W,3) → tensor (3,224,224)."""
        f0  = np.clip((frame_np[..., 0] + 1) / 2, 0, 1)
        f1  = np.clip((frame_np[..., 1] + 1) / 2, 0, 1)
        f2  = np.clip(frame_np[..., 2], 0, 1)
        rgb = (np.stack([f0, f1, f2], axis=-1) * 255).astype(np.uint8)
        return tr(_PIL_Image.fromarray(rgb))   # (3, 224, 224)

    def _flush_buf(buf, lab_buf, feat_buf, ext):
        if not buf:
            return
        with torch.no_grad():
            _t = torch.stack(buf).to(DEVICE)          # (N_buf, 3, 224, 224)
            _f = ext(_t).cpu().numpy()                # (N_buf, 2048)
        feat_buf.append(_f)

    for _si, (_spath, _slabel, _) in enumerate(dataset.samples):
        try:
            _seq = np.load(_spath)                     # (N_frames, H, W, 3)
        except Exception:
            continue
        _mid = _seq[len(_seq) // 2]                   # frame central
        _img_buf.append(_frame_to_tensor_single(_mid))
        _lab_buf.append(dataset.label_map[_slabel])

        if len(_img_buf) >= _SEP_BATCH:
            _flush_buf(_img_buf, _lab_buf, _feat_buf, _ext)
            _img_buf = []

        if (_si + 1) % 50 == 0:
            print(f"    {_si+1}/{len(dataset.samples)} procesadas...")

    _flush_buf(_img_buf, _lab_buf, _feat_buf, _ext)  # flush final
    del _ext, _img_buf

    _all_feats = np.concatenate(_feat_buf, axis=0)   # (N, 2048)
    _all_labs  = np.array(_lab_buf)
    del _feat_buf, _lab_buf
    print(f"  ✅ Features extraídos: {_all_feats.shape}")

    _inv_full  = {v: k for k, v in dataset.label_map.items()}
    _cls_list  = sorted(dataset.label_map.values())

    # Centroid y varianza intra-clase promediadas por dimensión
    _cents, _vars = {}, {}
    for _c in _cls_list:
        _mask = _all_labs == _c
        if not _mask.any():
            continue
        _fc        = _all_feats[_mask]
        _cents[_c] = _fc.mean(0)
        _vars[_c]  = float(_fc.var(0).mean())

    # FDR normalizado por dimensión
    _fdr = np.zeros((_nc_full, _nc_full))
    for _i in _cls_list:
        for _j in _cls_list:
            if _i == _j:
                continue
            _d2  = float(np.mean((_cents[_i] - _cents[_j]) ** 2))
            _den = _vars[_i] + _vars[_j] + 1e-8
            _fdr[_i, _j] = _d2 / _den

    # Imprimir matriz FDR
    _cnames = [_inv_full[c] for c in _cls_list]
    _w = 12
    print(f"\n  FDR(i,j) — mayor = más separable  |  >1.0 bueno, <0.5 problemático\n")
    print("  " + f"{'':>12}" + "".join(f"{n:>{_w}}" for n in _cnames))
    for _i in _cls_list:
        _row = f"{_inv_full[_i]:>12}"
        for _j in _cls_list:
            _row += f"{'—':>{_w}}" if _i == _j else f"{_fdr[_i,_j]:>{_w}.3f}"
        print(f"  {_row}")

    # Isolation Score por clase
    _iso = {
        _c: float(np.mean([_fdr[_c, _j] for _j in _cls_list if _j != _c]))
        for _c in _cls_list
    }
    _med_iso = float(np.median(list(_iso.values())))

    print(f"\n  Isolation Score (media FDR vs todas las demás clases)")
    print(f"  {'Clase':>12}  {'Score':>7}  {'n':>4}  Diagnóstico")
    print(f"  {'-'*54}")
    for _c in sorted(_cls_list, key=lambda x: _iso[x]):
        _n    = int((_all_labs == _c).sum())
        if _iso[_c] < _med_iso * 0.70:
            _diag = "⚠️  BAJA separabilidad → candidata a exclusión"
        elif _iso[_c] < _med_iso:
            _diag = "❌  por debajo de la mediana"
        else:
            _diag = "✅  aceptable"
        print(f"  {_inv_full[_c]:>12}  {_iso[_c]:>7.3f}  {_n:>4}  {_diag}")

    print(f"\n  Mediana isolation score: {_med_iso:.3f}")
    print(f"  Umbral exclusión: < {_med_iso*0.7:.3f}  (70% de la mediana)")

    del _all_feats, _all_labs, _cents, _vars, _fdr, _iso

# =========================
# EXCLUIR tristeza DEL MODELO FINAL
#
# Justificación combinada:
#   1) Isolation score más bajo del conjunto (tabla de separabilidad)
#   2) Solo 7-10 muestras primarias → 1-2 por fold de val → recall=0 en todos los folds
#   3) Solapamiento con asco documentado en la literatura de CASME II
# =========================
EXCLUDE_CLASSES = {"tristeza"}
_n_before = len(dataset.samples)
dataset.samples = [s for s in dataset.samples if s[1] not in EXCLUDE_CLASSES]
print(f"\n  ⚠️  Clases excluidas: {EXCLUDE_CLASSES}")
print(f"     {_n_before} → {len(dataset.samples)} muestras")

# Reconstruir label_map sin tristeza
labels_unique     = sorted(set(s[1] for s in dataset.samples))
dataset.label_map = {l: i for i, l in enumerate(labels_unique)}
num_classes       = len(dataset.label_map)

labels_all     = np.array([dataset.label_map[s[1]] for s in dataset.samples])
counts_primary = np.bincount(labels_all)

print(f"\nClases finales ({num_classes}): {dataset.label_map}")
print(f"Total muestras PRIMARIO: {len(dataset.samples)}")
print("Distribución por clase:")
for label, idx in dataset.label_map.items():
    n    = counts_primary[idx]
    src  = " (MEME_v2)" if label in _promoted_labels else ""
    warn = "  ⚠️  muy pocas muestras" if n < 10 else ""
    print(f"  {label}: {n} muestras{src}{warn}")

# =========================
# LOAD DATASET MEME (solo balanceo de train, nunca val)
# =========================
print("\nCargando dataset MEME (solo balanceo de clases en train)...")
dataset_meme = FlowDataset(
    data_dirs=[MEME_DATA_DIR],
    csv_files=[MEME_CSV_INDEX],
    transform=transform,
    num_frames=NUM_FRAMES,
    source_filters=[None]
)

dataset_meme.samples = [
    s for s in dataset_meme.samples
    if s[1] not in (_promoted_labels | {"otros"}) and s[1] in dataset.label_map
]
dataset_meme.label_map = dataset.label_map

labels_meme_all = np.array([dataset_meme.label_map[s[1]] for s in dataset_meme.samples])
counts_meme     = np.bincount(labels_meme_all, minlength=num_classes)
print(f"\nTotal muestras MEME disponibles (balanceo): {len(dataset_meme.samples)}")
print("Distribución MEME por clase (solo balanceo):")
for label, idx in dataset_meme.label_map.items():
    if counts_meme[idx] > 0:
        print(f"  {label}: {counts_meme[idx]} muestras")

# Verificar pipeline
x, y = dataset[0]
print(f"\nMuestra PRIMARIO OK: x={x.shape}, y={y}")


Cargando dataset PRIMARIO (CASME II + SMIC)...
  outputs_dataset_index: 255 cargadas
  output_extraction_smic: 0 cargadas (71 omitidas)

Etiquetas finales: {'asco': 0, 'felicidad': 1, 'miedo': 2, 'otros': 3, 'sorpresa': 4, 'tristeza': 5}

Clases finales (5): {'asco': 0, 'felicidad': 1, 'miedo': 2, 'sorpresa': 3, 'tristeza': 4}
Total muestras PRIMARIO: 129
Distribución por clase:
  asco: 63 muestras
  felicidad: 32 muestras
  miedo: 2 muestras  ⚠️  muy pocas muestras primarias
  sorpresa: 25 muestras
  tristeza: 7 muestras  ⚠️  muy pocas muestras primarias

Cargando dataset MEME (solo balanceo de clases en train)...
  output_extraction_meme_correction: 167 cargadas

Etiquetas finales: {'asco': 0, 'enojo': 1, 'felicidad': 2, 'miedo': 3, 'sorpresa': 4, 'tristeza': 5}

Total muestras MEME disponibles: 142
Distribución MEME por clase:
  asco: 33 muestras
  felicidad: 23 muestras
  miedo: 22 muestras
  sorpresa: 29 muestras
  tristeza: 35 muestras

Muestra PRIMARIO OK: x=torch.Size([16, 3, 2

In [ ]:
import torch.nn.functional as F

# =========================
# HYPERPARAMS MODELO
# =========================
PROJ_DIM = 256

_tmp = ResNet50FlowMLP(num_classes, proj_dim=PROJ_DIM).to(DEVICE)
trainable = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
total     = sum(p.numel() for p in _tmp.parameters())
print(f"🔧 ResNet50 (conv1-layer3 congelados) + layer4 (fine-tune) + Temporal Attention + MLP:")
print(f"   Parámetros entrenables: {trainable/1e6:.3f}M / {total/1e6:.2f}M")
del _tmp

print(f"\n   AdamW  lr={LR} (temporal_attn + classifier)   lr={LR_LAYER4} (layer4)")
print(f"   backbone conv1-layer3 congelados — sin gradientes en ~15M params")
print(f"   layer4 descongelada — fine-tune con LR_LAYER4={LR_LAYER4} (muy conservador)")
print(f"   CosineAnnealingLR  T_max={EPOCHS}")

# =========================
# CLASS WEIGHTS (con cap para evitar colapso por clases ultra-raras)
# =========================
labels_arr    = np.array([dataset.label_map[s[1]] for s in dataset.samples])
counts_all    = np.bincount(labels_arr)
class_weights = len(labels_arr) / (counts_all * len(counts_all))

class_weights = np.minimum(class_weights, MAX_CLASS_WEIGHT)

print(f"\n⚖️  Pesos de clases (Focal Loss, cap={MAX_CLASS_WEIGHT}x):")
for i, w in enumerate(class_weights):
    lbl = [l for l, idx in dataset.label_map.items() if idx == i][0]
    print(f"  Clase {i} ({lbl}): {w:.3f}x  (n={counts_all[i]})")

class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# =========================
# FOCAL LOSS (gamma=2, sin label_smoothing)
# =========================
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma: float = 2.0):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce    = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt    = torch.exp(-ce)
        focal = (1.0 - pt) ** self.gamma * ce
        return focal.mean()

criterion = FocalLoss(weight=class_weights_t, gamma=2.0)
print(f"\n🎯 Criterio: FocalLoss(gamma=2.0)  [sin label_smoothing — incompatibles]")


🔧 ResNet50 (totalmente congelado) + MLP clasificador:
   Parámetros entrenables: 0.559M / 24.07M

   AdamW  lr=0.0003  weight_decay=1e-2
   CosineAnnealingLR  T_max=60

⚖️  Pesos de clases (cross-entropy):
  Clase 0 (asco): 0.410x  (n=63)
  Clase 1 (felicidad): 0.806x  (n=32)
  Clase 2 (miedo): 12.900x  (n=2)
  Clase 3 (sorpresa): 1.032x  (n=25)
  Clase 4 (tristeza): 3.686x  (n=7)


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler

# =========================
# CONFUSION MATRIX
# =========================
def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# =========================
# MACRO F1 — solo sobre clases presentes en val
# =========================
def macro_f1(cm, active_classes=None):
    n = len(cm)
    if active_classes is None:
        active_classes = set(range(n))
    f1_scores = []
    for c in range(n):
        if c not in active_classes:
            continue
        if cm[c, :].sum() == 0:
            continue
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
    return sum(f1_scores) / len(f1_scores) if f1_scores else 0.0

# =========================
# EVALUATE
# =========================
@torch.no_grad()
def evaluate(model, loader, criterion, num_classes):
    model.eval()
    total_loss = correct = total = 0
    y_true, y_pred = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out  = model(x)
        loss = criterion(out, y)
        total_loss += loss.item() * y.size(0)
        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
    cm = confusion_matrix_np(y_true, y_pred, num_classes)
    active = set(int(c) for c in y_true)
    f1 = macro_f1(cm, active_classes=active)
    return total_loss / total, correct / total, f1, cm, active

# =========================
# STRATIFIED K-FOLD (sin sklearn)
# =========================
def make_kfold_splits(labels, n_splits, seed=42):
    rng      = np.random.default_rng(seed)
    classes  = np.unique(labels)
    fold_lists = [[] for _ in range(n_splits)]
    for c in classes:
        idx_c  = np.where(labels == c)[0]
        idx_c  = rng.permutation(idx_c).tolist()
        chunks = np.array_split(idx_c, n_splits)
        for k, chunk in enumerate(chunks):
            fold_lists[k].extend(chunk.tolist())
    splits = []
    for val_fold in range(n_splits):
        val_idx   = fold_lists[val_fold]
        train_idx = [i for k, fold in enumerate(fold_lists) for i in fold if k != val_fold]
        splits.append((train_idx, val_idx))
    return splits

labels      = np.array([dataset.label_map[s[1]] for s in dataset.samples])
num_classes = len(np.unique(labels))
splits      = make_kfold_splits(labels, N_FOLDS, seed=42)

inv_label_map = {i: l for l, i in dataset.label_map.items()}

print(f"✅ {N_FOLDS}-Fold CV estratificado  (total primario: {len(labels)})")
print(f"{'Fold':<6} {'Train':>6} {'Val':>5}  Distribución val por clase")
for fold_i, (tr, va) in enumerate(splits):
    va_dist = np.bincount([labels[i] for i in va], minlength=num_classes)
    dist_str = "  ".join(
        f"{inv_label_map[c]}={va_dist[c]}" + (" ⚠️" if va_dist[c] == 0 else "")
        for c in range(num_classes)
    )
    print(f"  {fold_i+1}    {len(tr):>5}   {len(va):>4}   {dist_str}")

# =========================
# K-FOLD TRAIN LOOP
# =========================
best_global_f1 = 0.0
fold_best_f1s  = []
patience       = 25   # revertido a 25: patience alto + epochs=100 deja correr más overfitting

for fold_i, (train_idx, val_idx) in enumerate(splits):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_i+1}/{N_FOLDS}")
    print(f"{'='*60}")

    val_classes_present = set(int(labels[i]) for i in val_idx)
    val_classes_absent  = set(range(num_classes)) - val_classes_present
    if val_classes_absent:
        absent_names = [inv_label_map[c] for c in sorted(val_classes_absent)]
        print(f"  ℹ️  Clases ausentes en val (excluidas del F1): {absent_names}")

    # ── MEME balancing con límite MAX_MEME_RATIO ──────────────────
    train_labels_primary = [labels[i] for i in train_idx]
    counts_train         = np.bincount(train_labels_primary, minlength=num_classes)

    meme_labels_arr    = np.array([dataset_meme.label_map[s[1]] for s in dataset_meme.samples])
    meme_by_class      = {c: list(np.where(meme_labels_arr == c)[0]) for c in range(num_classes)}
    meme_indices_added = []
    meme_labels_added  = []

    rng_fold = np.random.default_rng(seed=42 + fold_i)
    for c in range(num_classes):
        label_name   = inv_label_map[c]
        n_primary    = counts_train[c]
        needed       = max(0, MIN_TRAIN_SAMPLES - n_primary)
        meme_cap     = int(MAX_MEME_RATIO * max(n_primary, 1))
        to_add_max   = min(needed, meme_cap)
        available    = list(meme_by_class.get(c, []))
        rng_fold.shuffle(available)
        to_add = available[:to_add_max]
        meme_indices_added.extend(to_add)
        meme_labels_added.extend([c] * len(to_add))
        if len(to_add) > 0:
            print(f"  📊 {label_name}: {n_primary} prim + {len(to_add)} MEME → {n_primary+len(to_add)}"
                  + (f"  (cap={meme_cap})" if to_add_max < needed else ""))

    # ── Subsets ──────────────────────────────────────────────────
    train_primary_ds = AugmentedSubset(dataset, train_idx, train_transform)
    val_dataset_fold = Subset(dataset, val_idx)

    if meme_indices_added:
        train_meme_ds = AugmentedSubset(dataset_meme, meme_indices_added, train_transform)
        train_dataset = ConcatDataset([train_primary_ds, train_meme_ds])
    else:
        train_dataset = train_primary_ds

    all_train_labels = train_labels_primary + meme_labels_added
    counts_all_train = np.bincount(all_train_labels, minlength=num_classes)
    weights_s        = 1.0 / (counts_all_train + 1e-6)
    sample_weights   = [weights_s[l] for l in all_train_labels]
    sampler          = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=0, drop_last=True)
    val_loader   = DataLoader(val_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    n_meme = len(meme_indices_added)
    meme_pct = 100 * n_meme / max(len(train_dataset), 1)
    print(f"  Train: {len(train_dataset)} ({len(train_primary_ds)} prim + {n_meme} MEME [{meme_pct:.0f}%])  |  "
          f"Val: {len(val_dataset_fold)}")
    print(f"  Dist TRAIN: { {inv_label_map[c]: int(counts_all_train[c]) for c in range(num_classes)} }")

    # ── Modelo fresco por fold con seed fija (reproducibilidad) ───
    torch.manual_seed(42 + fold_i)   # fija inicialización: elimina varianza entre corridas
    model_fold = ResNet50FlowMLP(num_classes, proj_dim=PROJ_DIM).to(DEVICE)

    # layer4 con LR_LAYER4 muy bajo durante todo el entrenamiento
    optimizer_fold = torch.optim.AdamW([
        {"params": model_fold.layer4.parameters(),        "lr": LR_LAYER4, "weight_decay": 1e-4},
        {"params": model_fold.temporal_attn.parameters(), "lr": LR,        "weight_decay": 5e-2},
        {"params": model_fold.classifier.parameters(),    "lr": LR,        "weight_decay": 5e-2},
    ])
    scheduler_fold = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_fold, T_max=EPOCHS, eta_min=1e-6,
    )

    best_f1_fold = 0.0
    counter_fold = 0

    for epoch in range(EPOCHS):
        model_fold.train()
        train_loss = correct = total = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer_fold.zero_grad()
            out  = model_fold(x)
            loss = criterion(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_fold.parameters(), max_norm=1.0)
            optimizer_fold.step()
            train_loss += loss.item() * y.size(0)
            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            total   += y.size(0)

        train_loss /= total
        train_acc   = correct / total

        val_loss, val_acc, val_f1, cm, _ = evaluate(model_fold, val_loader, criterion, num_classes)
        scheduler_fold.step()

        if val_f1 > best_f1_fold:
            best_f1_fold = val_f1
            counter_fold = 0
            if val_f1 > best_global_f1:
                best_global_f1 = val_f1
                torch.save({
                    "model_state_dict": model_fold.state_dict(),
                    "label_map": dataset.label_map,
                    "fold": fold_i + 1,
                    "epoch": epoch + 1,
                }, "best_resnet50_mlp.pth")
                print(f"  💾 [Fold {fold_i+1} | Ep {epoch+1}] "
                      f"NUEVO MEJOR GLOBAL F1={best_global_f1:.4f}")
        else:
            counter_fold += 1

        if (epoch + 1) % 5 == 0:
            print(f"  [F{fold_i+1} | Ep {epoch+1:3d}] "
                  f"TrL={train_loss:.3f} TrAcc={train_acc:.3f} | "
                  f"VL={val_loss:.3f} VAcc={val_acc:.3f} F1={val_f1:.3f} "
                  f"(best={best_f1_fold:.3f})")

        if counter_fold >= patience:
            print(f"  ⛔ Early stopping en epoch {epoch+1}")
            break

    fold_best_f1s.append(best_f1_fold)
    _, _, _, cm_final, active = evaluate(model_fold, val_loader, criterion, num_classes)

    print(f"\n  ✅ Fold {fold_i+1} finalizado. Best Val F1: {best_f1_fold:.4f}")
    print(f"  Clases evaluadas: {[inv_label_map[c] for c in sorted(active)]}")
    header = "       " + "  ".join(f"{inv_label_map[c]:>9}" for c in range(num_classes))
    print(f"  Confusion Matrix:\n  {header}")
    for r in range(num_classes):
        if cm_final[r, :].sum() == 0:
            continue
        row_str = "  ".join(f"{cm_final[r, c]:>9}" for c in range(num_classes))
        print(f"  {inv_label_map[r]:>8} [{row_str}]")

# =========================
# RESUMEN CROSS-VALIDATION
# =========================
fold_arr = np.array(fold_best_f1s)
print(f"\n{'='*60}")
print(f"🏆 {N_FOLDS}-FOLD CROSS-VALIDATION COMPLETADO")
print(f"   F1 por fold: {[f'{f:.4f}' for f in fold_best_f1s]}")
print(f"   F1 media   : {fold_arr.mean():.4f} ± {fold_arr.std():.4f}")
print(f"   Nota: F1 calculado solo sobre clases presentes en cada fold de val")
print(f"   Mejor modelo global guardado: best_resnet50_mlp.pth  (F1={best_global_f1:.4f})")


✅ 5-Fold CV estratificado  (total primario: 129)
Fold    Train   Val  Distribución val por clase
  1      101     28   asco=13  felicidad=7  miedo=1  sorpresa=5  tristeza=2
  2      101     28   asco=13  felicidad=7  miedo=1  sorpresa=5  tristeza=2
  3      104     25   asco=13  felicidad=6  miedo=0 ⚠️  sorpresa=5  tristeza=1
  4      105     24   asco=12  felicidad=6  miedo=0 ⚠️  sorpresa=5  tristeza=1
  5      105     24   asco=12  felicidad=6  miedo=0 ⚠️  sorpresa=5  tristeza=1

  FOLD 1/5
  📊 felicidad: 25 prim + 5 MEME → 30
  📊 miedo: 1 prim + 3 MEME → 4  (cap=3)
  📊 sorpresa: 20 prim + 10 MEME → 30
  📊 tristeza: 5 prim + 15 MEME → 20  (cap=15)
  Train: 134 (101 prim + 33 MEME [25%])  |  Val: 28
  Dist TRAIN: {'asco': 50, 'felicidad': 30, 'miedo': 4, 'sorpresa': 30, 'tristeza': 20}
  💾 [Fold 1 | Ep 1] NUEVO MEJOR GLOBAL F1=0.0875
  💾 [Fold 1 | Ep 5] NUEVO MEJOR GLOBAL F1=0.1016
  [F1 | Ep   5] TrL=0.864 TrAcc=0.410 | VL=2.456 VAcc=0.143 F1=0.102 (best=0.102)
  [F1 | Ep  10] TrL=0.

## Resultados — Mejor ejecución (F1 media = 0.5203, mejor fold = 0.6087)

### Configuración que produjo estos resultados

| Parámetro | Valor |
|---|---|
| `NUM_FRAMES` | 16 |
| `BATCH_SIZE` | 8 |
| `EPOCHS` | 100 |
| `N_FOLDS` | 5 |
| `LR` (temporal_attn + classifier) | 3e-4 |
| `LR_LAYER4` (fine-tuning layer4) | 1e-5 |
| `MAX_CLASS_WEIGHT` | 8.0 |
| `MIN_TRAIN_SAMPLES` | 30 |
| `MAX_MEME_RATIO` | 5.0 |
| Scheduler | CosineAnnealingLR (T_max=100, eta_min=1e-6) |
| Grad clip | max_norm=1.0 |
| patience | 25 |
| Dropout | 0.55 / 0.45 |
| Seed por fold | torch.manual_seed(42 + fold_i) |

### Arquitectura del modelo

- **Backbone**: ResNet50 pre-entrenado (ImageNet)
  - `conv1 → layer3`: **CONGELADO** (sin gradientes, ~15M params)
  - `layer4`: **DESCONGELADO** todo el entrenamiento con `LR_LAYER4 = 1e-5`
- **Temporal Attention**: aprende a ponderar los 16 frames por relevancia discriminativa
- **MLP Clasificador** (`PROJ_DIM=256`):  
  `2048 → 256 (BN+ReLU+Drop 0.55) → 128 (BN+ReLU+Drop 0.45) → 5 clases`
- **Loss**: FocalLoss(γ=2.0) con class weights (cap=8.0), sin label_smoothing
- **Optimizer**: AdamW con 3 grupos de LR (layer4 / temporal_attn / classifier)

### Dataset
- **Primario** (train + val): CASME II (todas las emociones) + SMIC (solo `sorpresa`)
  + neutral y miedo promovidos desde MEME_v2 → **257 muestras primarias, 5 clases**
- **Clases**: asco, felicidad, miedo, neutral, sorpresa (tristeza excluida por baja separabilidad FDR)
- **MEME_v2** (solo balanceo de train): `MIN_TRAIN_SAMPLES=30` → solo ~4-5 muestras extra para felicidad

### Resultados 5-Fold CV

| Fold | Best Val F1 | Val(n) | Early stop |
|---|---|---|---|
| 1 | 0.4467 | 53 | Ep 63 |
| 2 | 0.4610 | 53 | Ep 86 |
| 3 | **0.6087** | 52 | Ep 100 |
| 4 | 0.5794 | 51 | Ep 82 |
| 5 | 0.5060 | 48 | Ep 79 |

**F1 media: 0.5203 ± 0.0640**  
**Mejor modelo global**: `best_resnet50_mlp.pth` (F1 = 0.6087, Fold 3, Epoch 79)

### Mejoras clave (historial)

1. **FocalLoss(γ=2.0)** — sin label_smoothing (incompatibles)
2. **layer4 descongelada** con LR_LAYER4=1e-5 (todo el entrenamiento): +0.10 F1 vs layer4 congelada
3. **neutral y miedo promovidos** al pool primario desde MEME_v2 → 257 muestras (antes 235)
4. **tristeza excluida** — isolation score FDR más bajo + solo 1-2 muestras por fold de val
5. **MIN_TRAIN=30** (no 50): más MEME → domain shift severo, peor F1
6. **Dropout 0.55/0.45** — con 0.65/0.55 el modelo subajusta (TrAcc 0.70, F1 val igual o peor)
7. **torch.manual_seed(42 + fold_i)** — inicialización reproducible por fold
